# 68 · PT species level as a positive control: does the constant human–mouse difference recover known differences?

**The question.** Notebooks 37–67 analyse the *shape* of human-vs-mouse differences along the PT and leave the
constant *level* out, for two reasons:
- **Probe efficiency.** Each gene's probe efficiency depends on its panel, and the human and mouse panels differ
  (SI, "Probe panels").
- **Replication.** With one human donor and two mice there is no species-level replication.

This notebook is a **positive control of the measurement, not a discovery screen**. It asks whether the level
still recovers species differences that are already known:
- curated literature controls;
- an external snRNA atlas comparison, Lake/KPMP human cortex against CELLxGENE Census mice;
- genome-defined paralog families with unequal member counts.

It reports agreement and effect sizes. **No species-level p-value is computed on our data.**

**The frozen protocol.** Everything below follows `docs/results/pt-gene-model-nb.md`, section "Notebook 68
protocol". It was committed in aa15069, before any level statistic was computed on our data or the atlases, and
its readings are fixed there. In short:
- **L_g**: (1 / ln 2) × the uniform grid mean of [Δ_g(x) − m(x)].
  - Δ is the NB natural-log rate ratio, human over mouse, from the shared gene model of notebooks 66 and 67 on
    SCF13.
  - m(x) is the median of Δ over the reference genes at each grid point.
- **P_g**: the same contrast from equal-weight S1/S2/S3 pseudobulk log-CPM. It does not use the coordinate, and
  it gives the direction of separated genes.
- **A_g**: limma-trend on donor-level equal-segment log-CPM of the atlas, median-centred.
  - Human: Lake, 7 cortex donors.
  - Mouse: Census, 6 adult males.
- **Criteria.**
  - (i) ≥ 80% of the literature gene controls verified for mouse (mostly protein abundances, Thakur et al.
    2024) have the correct sign in our universe.
  - (ii) ≥ 80% sign agreement with A among genes that are atlas-strong (BH ≤ 0.05, |A| ≥ 1) and have |L| ≥ 1.
  - (iii) and (iv) are descriptive.
  - "Level recovers known biology" if (i) and (ii) both pass.

**Summary (numbers from this run).**
- **(i) fails.** 17 of 27 primary literature controls have the expected sign (63%, Wilson 44–78%); 8 clear the
  housekeeping band.
  - The controls are protein abundances. The atlas (mRNA) agrees with them for only 19 of the 27.
  - Of our 10 failures, 6 fail in the atlas too: Abcc4, Fmo1, Slc16a1, Slc22a6, Slc22a7 and Slc22a8.
  - Only Abcc2 is a clear failure of our measurement: the atlas has it strongly mouse-higher (A = −2.6), while
    our L is +0.6. Gusb is small on both sides.
- **(ii) passes.** 3,584 of 3,984 atlas-strong genes with |L| ≥ 1 share the atlas sign (90.0%).
  - Every sensitivity gives 88.6–91.6%.
  - All atlas-strong genes regardless of |L| give 81.0%.
- **Reading (fail, pass).** Level agrees with the atlas genome-wide but not with the curated controls, which
  are protein measurements. The overall criterion is not met.
- **(iii).** Spearman L vs A is 0.60 [0.58, 0.61], and 0.62 on equal-probe genes, against an atlas-to-atlas
  ceiling of 0.97.
  - L − A rises with log2(human probes / mouse probes) over the unbalanced genes (ρ = 0.45), so the probe count
    leaks into the level.
  - The housekeeping band is about as wide in our data as in the atlas (5.7 against 5.4 log2).
- **(iv) Families.**
  - The member-count expectation fails in both datasets: 198 of 410 families in our data (48%) and 52% in the
    atlas.
  - F agrees with the atlas F in sign for 73% of families (ρ = 0.67).
  - The pre-named rodent expansions are mouse-higher in both: Slco1a, Cyp2d, Cyp2j, Ces1, Akr1c and Nat8.
  - Sult1a disagrees, with F −4.2 against an atlas F of +4.5. Its two human members have one probe each and are
    detected in < 0.5% of human structures.
- **Pathways.** Mean L vs mean A has Spearman 0.70 across the 1,513 sets.

**Sections:**
1. Setup, frozen inputs and annotations.
2. The NB species gene model.
3. The atlas reference.
4. Our level: L, the pair spread, the same-species reference and P.
5. Bands: the housekeeping yardstick.
6. Literature controls: criterion (i).
7. Atlas agreement: criteria (ii) and (iii), with probe and pairing sensitivities.
8. Families: (iv).
9. Pathways (descriptive).
10. Criteria and reading.
11. Main figure.
12. Post hoc (added after the criteria were seen).
13. Run record.

**Assumptions and caveats.**
- **Replicates.** Specimens are the replicates on our side and donors in the atlases. The two human "replicates"
  are two cortex sections of **one donor**, so the pair spread covers sections and mice, not human donors.
- **Probe efficiency.** Centring at each grid point removes depth, library composition, the mitochondrial share
  and any efficiency difference shared by a whole panel. It **cannot remove gene-specific probe efficiency**.
  That is why this is a positive control and every gene-level level difference stays descriptive.
- **The atlases.**
  - In the atlases, species is also confounded with dataset: lab, protocol and donor ages. Agreement with
    the atlas therefore means **consistency across technologies, not proof of biology**.
  - snRNA measures nuclear RNA and Visium HD whole-cell RNA. A nuclear-fraction bias that is conserved between
    species cancels in the human-vs-mouse difference.
- **Sex.** Lake's 7 donors include 4 women, while all our specimens are male (sensitivity: Lake's 3 men).
- **The coordinate** is treated as exact.

## 1 · Setup

In [ ]:
import argparse
import hashlib
import json
import os
import re
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
external = data_root / 'external'
coordinate_file = results_root / 'pt_reconstruction_v2' / 'primary_pt_coordinate.csv'
upstream13 = results_root / 'minimal_pt_scfates'
nb38 = results_root / 'pt_literature_deep'                  # read only: probe table; Lake cache (guard)
nb62 = results_root / 'pt_non_one_to_one_orthologs'         # read only: orthogroups and inventories; Lake cache (guard)
nb64 = results_root / 'pt_paralog_pairing_check'            # read only: pairs inside non-1:1 orthogroups
nb67 = results_root / 'pt_normalized_t_spatial'             # read only: the species gene model (guard)
LAKE = external / 'kpmp_lake2023' / 'lake2023_snRNA_v1.5.h5ad'
CENSUS = external / 'census_pt_segments'
HOUSEKEEPING = external / 'housekeeping' / 'HK_genes.txt'
LITERATURE = project / 'docs' / 'results' / 'pt-level-literature-controls.csv'
MOUSE, HUMAN = ['Ctrl1A2', 'Ctrl1A4'], ['HUK1_COR1', 'HUK1_MED1']
MATRICES = {s: data_root / 'tubule_by_gene' / f'{s}_tubule_by_gene_caleb.h5ad' for s in MOUSE + HUMAN}
output = results_root / 'pt_species_level_control'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)
REQUIRED = [coordinate_file, upstream13 / 'cross_species_pt_scfates.h5ad', upstream13 / 'ortholog_map_used.csv',
            LAKE, HOUSEKEEPING, LITERATURE, *MATRICES.values(), nb38 / 'probe_counts.csv',
            nb64 / 'tables' / 'dropped_pairs.csv',
            *[nb62 / 'tables' / f for f in ('family_catalogue.csv', 'gene_inventory_mouse.csv', 'gene_inventory_human.csv')],
            *[CENSUS / f for f in ('mouse_pt_segment_pseudobulk_counts.csv.gz', 'mouse_pt_segment_pseudobulk_meta.csv',
                                   'human_pt_segment_pseudobulk_counts.csv.gz', 'human_pt_segment_pseudobulk_meta.csv',
                                   'mouse_pt_segment_pseudobulk_counts_families.csv.gz',
                                   'mouse_pt_segment_pseudobulk_meta_families.csv', 'mouse_family_genes_fetched.csv',
                                   'census_symbol_audit.csv')]]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (notebooks 13, 38, 62 and 64; '
                                'analysis/scripts/fetch_census_pt_*.py; the frozen literature and housekeeping files)')

NOTEBOOK_LOGIC_VERSION = '68.level.1'   # Bump when a cached calculation changes.
# Frozen inputs (protocol): the guards in 1.1 refuse to run on any other file.
LITERATURE_SHA256 = 'f1aa71844552327e7f8b11905167fae34c205398c2499053711bcbee2ca06c4f'   # commit aa15069
HOUSEKEEPING_SHA256 = 'df7aa10e635efb5b0df7dd607e5c11c91df2bdf3e321fb97c0c088ea96a60756'
EXPECTED_SIGN = {'higher in human': 1, 'higher in mouse': -1}
# Our cohort and the shared gene model (notebooks 37, 66, 67)
EXPECTED_STRUCTURES, EXPECTED_GENES, EXPECTED_SETS = 12272, 11071, 1513
BASIS_DF, GRID_POINTS = 6, 101
SEGMENTS = ('S1', 'S2', 'S3')
# Atlas reference
MIN_NUCLEI = 50
LAKE_DONORS = ['164-10', '164-6', '18-142', '18-312', '3535', 'KRP446', 'KRP460']
ADULT_STAGES = ('3-6 month-old mature stage', '12-month-old stage')
N_ADULT_MALES, N_MALES, N_CENSUS_HUMAN = 6, 12, 6
CENSUS_SEGMENTS = {'epithelial cell of proximal tubule segment 1': 'S1', 'epithelial cell of proximal tubule segment 2': 'S2',
                   'epithelial cell of proximal tubule segment 3': 'S3'}
CENSUS_HUMAN_SEGMENTS = {'kidney proximal convoluted tubule epithelial cell': 'PCT',
                         'epithelial cell of proximal tubule segment 3': 'S3'}
CENSUS_HUMAN_WEIGHTS = (2 / 3, 1 / 3)    # convoluted PT stands for S1 and S2, so S1, S2 and S3 weigh equally
ATLAS_FDR, ATLAS_LFC = .05, 1.           # atlas-strong
# Criteria
LEVEL_MIN = 1.                           # (ii): |L| >= 1
PASS_SHARE = .80
MIN_LITERATURE, MIN_ATLAS_GENES = 10, 30
BAND_QUANTILES = (.025, .975)
N_BOOT, SEED = 2000, 68
ABSENT = .01                             # nearly absent: detected in < 1% of one species' structures
# Families (notebook 62's orthogroups)
FAMILY_CLASSES = ('1:many', 'many:1', 'many:many')
FAMILY_DETECTION = .02
PRENAMED = {'Cyp4a': 'Cyp4a10', 'Akr1c': 'Akr1c6', 'Slco1a': 'Slco1a1', 'Sult2a': 'Sult2a1', 'Cyp2d': 'Cyp2d22',
            'Ces1': 'Ces1d', 'Nat8': 'Nat8f1', 'Cyp2j': 'Cyp2j5', 'Cyp2c': 'Cyp2c29', 'Sult1a': 'Sult1a1',
            'Ugt2b': 'Ugt2b5'}           # family -> a mouse member that names its orthogroup
N_JOBS = int(os.environ.get('PSEUDOSPACE_N_JOBS', '36'))   # parallel workers; never part of a cache key

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from importlib.metadata import version
from IPython.display import display

MM = 1 / 25.4
INK, MUTED, LIGHT, GREY = '#0b0b0b', '#52514e', '#d9d8d4', '#a8a7a2'
AGREE_COLOR, DISAGREE_COLOR = '#2a78d6', '#eb6834'   # validated categorical pair (CVD-safe); markers differ too
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.max_rows', 80)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')


print('Results folder:', output, '· parallel workers:', N_JOBS)

### 1.1 · Frozen inputs

The literature controls and the housekeeping list are read only if they are byte-identical to the files named in
the protocol. This guard is never cached.

In [ ]:
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


guard_log = []
if sha256(LITERATURE) != LITERATURE_SHA256:
    raise ValueError(f'{LITERATURE.name} differs from the frozen literature controls.')
if sha256(HOUSEKEEPING) != HOUSEKEEPING_SHA256:
    raise ValueError(f'{HOUSEKEEPING} differs from the frozen housekeeping list.')
guard_log.append('literature controls and housekeeping list match their frozen sha256')
print(guard_log[-1])

### 1.2 · Our structures and genes, and per-gene annotations

Notebook 37's construction on SCF13 (`coordinate_inputs.pt_inputs`), as in notebooks 66 and 67:
- **Counts.** Raw counts on the 11,071 one-to-one genes.
- **Library.** The total over the one-to-one orthologs measured in all specimens.
- **Gene sets.** The 1,513 pathway sets.

Per-gene annotations:
- detection in each species, and the *nearly absent* flag (< 1% of one species' structures);
- probe counts and the equal-probe flag (notebook 38);
- probe-clean status in both species (notebook 62);
- housekeeping membership, mapped by human symbol through the ortholog map;
- pairs inside non-1:1 orthogroups (notebook 64's dropped pairs);
- the Census symbol audit of `analysis/scripts/fetch_census_pt_families.py`.

Census columns whose symbol belongs to another Ensembl id than our panel's (`other_id`) measured another gene.
They are dropped from that atlas.

In [ ]:
import anndata as ad
from scipy import sparse

from pseudospace.coordinate_inputs import pt_inputs, read_coordinate
from pseudospace.stage_cache import cached_payload, digest

t0 = time.perf_counter()
hm = pt_inputs(results_root, data_root, read_coordinate(coordinate_file))
assert (len(hm['position']), len(hm['genes']), len(hm['gene_sets'])) == (EXPECTED_STRUCTURES, EXPECTED_GENES, EXPECTED_SETS), \
    "notebook 37's universe changed"
GENES = pd.Index(hm['genes'], name='gene')
counts = sparse.csr_matrix(hm['counts'], dtype=float)
library = np.asarray(hm['library'], float)
position = np.asarray(hm['position'], float)
specimen = np.asarray(hm['specimen']).astype(str)
segment = pd.Series(hm['segment']).astype(str).str.replace('PT-', '', regex=False).to_numpy()
human = np.asarray(hm['human'], bool)
structure_ids = hm['obs'].index.astype(str).to_numpy()
GENE_SETS = hm['gene_sets']
del hm
assert set(segment) == set(SEGMENTS) and set(specimen[human]) == set(HUMAN) and set(specimen[~human]) == set(MOUSE)
assert counts.shape == (len(position), len(GENES)) and len(structure_ids) == len(position)

orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
to_human = dict(zip(orthologs.mouse_symbol, orthologs.human_symbol))
to_mouse = dict(zip(orthologs.human_symbol.str.upper(), orthologs.mouse_symbol))
human_symbol = pd.Series(GENES.map(to_human), index=GENES)
probes = pd.read_csv(nb38 / 'probe_counts.csv').set_index('mouse_symbol')
inventory = {sp: pd.read_csv(nb62 / 'tables' / f'gene_inventory_{sp}.csv',
                             usecols=['gene', 'gene_id', 'component', 'ortholog_class', 'probe_clean', 'n_human', 'n_mouse'])
             for sp in ('mouse', 'human')}
dropped_pairs = set(pd.read_csv(nb64 / 'tables' / 'dropped_pairs.csv').mouse_symbol)
housekeeping = set(pd.read_csv(HOUSEKEEPING, sep=r'\s+', header=None, names=['symbol', 'refseq']).symbol.str.upper())
audit = pd.read_csv(CENSUS / 'census_symbol_audit.csv')
census_other = {sp: set(audit.column[audit.species.eq(sp) & audit.status.eq('other_id')]) for sp in ('mouse', 'human')}

detection = {side: np.asarray((counts[mask] > 0).mean(axis=0)).ravel() for side, mask in (('human', human), ('mouse', ~human))}
specimen_totals = np.array([np.asarray(counts[specimen == s].sum(axis=0)).ravel() for s in MOUSE + HUMAN])
clean = {sp: inventory[sp].assign(key=inventory[sp].gene.str.upper()).set_index('key').probe_clean.astype(bool)
         for sp in ('mouse', 'human')}
INFO = pd.DataFrame({
    'human_symbol': human_symbol,
    'detection_human': detection['human'], 'detection_mouse': detection['mouse'],
    'nearly_absent': np.minimum(detection['human'], detection['mouse']) < ABSENT,
    'zero_in_a_specimen': (specimen_totals == 0).any(axis=0),
    'mouse_probes': probes.mouse_probes.reindex(GENES).to_numpy(),
    'human_probes': probes.human_probes.reindex(GENES).to_numpy(),
    'equal_probes': probes.probe_balance.reindex(GENES).eq('balanced').to_numpy(),
    'probe_clean_both': (clean['mouse'].reindex(GENES.str.upper()).fillna(False).to_numpy(bool)
                         & clean['human'].reindex(human_symbol.str.upper()).fillna(False).to_numpy(bool)),
    'housekeeping': human_symbol.str.upper().isin(housekeeping).to_numpy(),
    'non_one_to_one_pair': GENES.isin(dropped_pairs),
    'census_mouse_status': audit[audit.species.eq('mouse')].set_index('column').status.reindex(GENES).fillna('absent').to_numpy(),
    'census_human_status': audit[audit.species.eq('human')].set_index('column').status.reindex(GENES).fillna('absent').to_numpy(),
}, index=GENES)
INPUT_TABLE = pd.Series({
    'structures': len(position),
    **{f'{s} structures': int((specimen == s).sum()) for s in MOUSE + HUMAN},
    'genes': len(GENES), 'pathways': len(GENE_SETS),
    'nearly absent in one species': int(INFO.nearly_absent.sum()),
    'zero counts in a specimen': int(INFO.zero_in_a_specimen.sum()),
    'equal probe counts': int(INFO.equal_probes.sum()),
    'equal probe counts and probe-clean in both': int((INFO.equal_probes & INFO.probe_clean_both).sum()),
    'housekeeping (Eisenberg & Levanon 2013)': int(INFO.housekeeping.sum()),
    'pairs inside non-1:1 orthogroups (notebook 64)': int(INFO.non_one_to_one_pair.sum()),
    'Census mouse symbol: other gene': int(INFO.census_mouse_status.eq('other_id').sum()),
    'Census human symbol: other gene': int(INFO.census_human_status.eq('other_id').sum())}, name='value')
INPUT_TABLE.to_csv(tables / 'inputs.csv')
display(INPUT_TABLE.to_frame())
print(f'Inputs {time.perf_counter() - t0:.0f} s')

## 2 · The NB species gene model

This is the shared gene model of notebooks 66 and 67 (`pathway_pipelines.standard_gene_model`) for the
human-vs-mouse species split:
- an NB GLM on raw counts with offset log(library);
- a 6-df cubic B-spline basis;
- prior weights wᵢ = n / (2·J·n_j);
- one α per gene from M_full.

It returns:
- **Group curves.** Equal-specimen means on a 101-point grid over the specimens' common [1st, 99th] percentile
  support.
- **Δ = human − mouse**, a natural-log rate ratio.
- **One curve per specimen**, from a separate per-specimen fit.

The model has **no grid weights**. Each grid point gets 1/101, as in notebook 67's `log_ratio`.

**Guard (not cached).** The refit must reproduce notebook 67's cached species model (rtol 1e-6):
- `T_spatial`, `T_level` and the flags;
- α, Δ and the group curves;
- the specimen curves.

In [ ]:
from pseudospace import nb_gam, pathway_calibration, pathway_pipelines, species_level, stats_gam
from pseudospace.pathway_pipelines import GeneModel, standard_gene_model

cache = output / 'stage_cache'
code_key = digest([NOTEBOOK_LOGIC_VERSION] + [Path(m.__file__) for m in
                                               (nb_gam, pathway_calibration, pathway_pipelines, species_level, stats_gam)])
SECONDS = {}   # uncached compute time of every cached stage


def cached(stage, compute, *, params=None, inputs=None):
    """cached_payload under this notebook's root and code; keeps the stage's uncached compute time."""
    def wrapped():
        t = time.perf_counter()
        out = dict(compute())
        out['__seconds'] = np.array(time.perf_counter() - t)
        return out
    result = cached_payload(stage, wrapped, root=cache, params=params, inputs=inputs, code=code_key)
    SECONDS[stage] = float(result.pop('__seconds'))
    return result


def model_payload(model):
    out = {f'stat|{c}': model.statistics[c].to_numpy() for c in model.statistics}
    out.update(grid=model.grid, knots=model.knots, reference=model.reference, case=model.case, delta=model.delta,
               se=model.se, group_names=model.groups.index.to_numpy().astype(str), group_values=model.groups.to_numpy(),
               alpha=np.asarray(model.dispersion['alpha'], float), alpha_converged=np.asarray(model.dispersion['converged'], bool))
    for name, curve in model.specimen_curves.items():
        out[f'curve|{name}'] = curve
        out[f'converged|{name}'] = model.specimen_converged[name].to_numpy()
        out[f'separated|{name}'] = model.specimen_separated[name].to_numpy()
    return out


def model_from(payload, genes):
    index = pd.Index(genes, name='gene')
    stats = pd.DataFrame({c: payload[f'stat|{c}'] for c in ('T_level', 'T_spatial', 'T_total', 'alpha', 'converged',
                                                            'separated')}, index=index)
    stats[['converged', 'separated']] = stats[['converged', 'separated']].astype(bool)
    names = sorted(k.split('|', 1)[1] for k in payload if k.startswith('curve|'))
    return GeneModel(statistics=stats, grid=payload['grid'], knots=payload['knots'], reference=payload['reference'],
                     case=payload['case'], delta=payload['delta'], se=payload['se'],
                     specimen_curves={n: payload[f'curve|{n}'] for n in names},
                     specimen_converged=pd.DataFrame({n: payload[f'converged|{n}'].astype(bool) for n in names}, index=index),
                     specimen_separated=pd.DataFrame({n: payload[f'separated|{n}'].astype(bool) for n in names}, index=index),
                     groups=pd.Series(payload['group_values'].astype(int), index=payload['group_names'].astype(str)),
                     dispersion={'alpha': payload['alpha'], 'converged': payload['alpha_converged'].astype(bool)})


group = human.astype(float)
model_inputs = {'counts': digest(counts), 'library': library, 'position': position, 'specimen': specimen,
                'genes': list(GENES), 'group': group}
payload = cached('gene_model_species', lambda: model_payload(standard_gene_model(
    counts, library, position, group, specimen, genes=list(GENES), basis_df=BASIS_DF, n_jobs=N_JOBS)),
    params={'basis_df': BASIS_DF, 'grid_points': GRID_POINTS, 'label': 'species'}, inputs=model_inputs)
MODEL = model_from(payload, GENES)
print(f"Gene model: {len(GENES):,} genes · grid {MODEL.grid[0]:.3f}–{MODEL.grid[-1]:.3f} ({len(MODEL.grid)} points) · "
      f"not converged {int((~MODEL.statistics.converged).sum())} · separated {int(MODEL.statistics.separated.sum())}")

In [ ]:
def newest_payload(folder, stage):
    """Another notebook's newest cached payload of one stage (read only), or None when its cache was purged."""
    files = sorted(Path(folder).glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        return None
    with np.load(files[-1], allow_pickle=False) as stored:
        return {k: stored[k] for k in stored.files}


assert len(MODEL.grid) == GRID_POINTS and np.allclose(np.diff(MODEL.grid), np.diff(MODEL.grid)[0])
stored67 = newest_payload(nb67 / 'stage_cache', 'gene_model_human_vs_mouse_species')
if stored67 is None:
    guard_log.append("notebook 67's species gene model is not cached: its reproduction guard was skipped")
else:
    for key in ('stat|T_spatial', 'stat|T_level', 'alpha', 'grid', 'delta', 'reference', 'case',
                *[f'curve|{n}' for n in MOUSE + HUMAN]):
        np.testing.assert_allclose(payload[key], stored67[key], rtol=1e-6, atol=1e-6, err_msg=key)
    for key in ('stat|converged', 'stat|separated', *[f'{k}|{n}' for k in ('converged', 'separated') for n in MOUSE + HUMAN]):
        np.testing.assert_array_equal(payload[key].astype(bool), stored67[key].astype(bool), err_msg=key)
    guard_log.append("species gene model reproduces notebook 67's cache (statistics, flags, alpha, Delta, curves; rtol 1e-6)")
del stored67
print(guard_log[-1])

## 3 · The atlas reference

These are single-nucleus RNA-seq atlases with poly-A capture and no probes. Donors are the replicates.

- **Human: Lake et al. 2023 (KPMP v1.5).**
  - Normal cortex nuclei (`region` C, `tissue` "cortex of kidney") with author labels PT-S1/S2/S3.
  - The 7 donors with ≥ 50 nuclei in every segment.
  - The pseudobulk is rebuilt here at the Ensembl level and mapped to our mouse symbols by notebook 38's rule:
    Ensembl id first, symbol second, duplicates summed.
  - **Guards:** it equals notebook 62's cortex cache (all genes) and notebook 38's cache on our genes.
- **Mouse: CELLxGENE Census 2025-11-08, dataset 25818bf7, segments 1/2/3.**
  - Primary: **adult males (3–6 months and 12 months), 6 donors** with ≥ 50 nuclei in every segment.
  - Sensitivity: all 12 males, including the prepubertal 21-day and the 21-month donors.
  - The family members come from `fetch_census_pt_families.py`. **Guards:** its groups and totals equal the main
    file's, and its controls are identical except the audited `other_id` columns.
- **Sensitivities.**
  - Lake's 3 male donors.
  - Census human cortex (dataset 09b518f9, the 6 donors with ≥ 50 nuclei in both labels) in place of Lake.
    Its convoluted PT is weighted 2/3 and S3 1/3.

**Per donor:**
- log-CPM = log2((y + 0.5) / (N + 1) · 10⁶) per segment, where N is the donor-segment total over the analysed
  genes.
- Then the equal-weight mean over segments.

**Analysed genes:** our universe genes measured in both atlases. Lake must map them, and Census must list them
under our gene.

**limma-trend.**
- `lmFit(~ species)`, then `eBayes(trend = TRUE, robust = TRUE)`.
- A = coefficient − its median.
- Tested against the median: t = A / (stdev.unscaled · √s²_post), with BH over the analysed genes.
- **Atlas-strong:** BH ≤ 0.05 and |A| ≥ 1.

In [ ]:
import h5py

from pseudospace.replication_rules import csr_group_sums
from pseudospace.species_level import (band, bootstrap_spearman, clears_band, equal_segment_log_cpm, family_sums,
                                       grid_centred_level, limma_trend_species, pair_levels, pair_summary,
                                       segment_pseudobulk, species_difference, wilson_interval)


def read_obs_column(group_, name):
    item = group_[name]
    if isinstance(item, h5py.Group):
        categories = np.array([c.decode() if isinstance(c, bytes) else str(c) for c in item['categories'][:]])
        return pd.Categorical.from_codes(item['codes'][:], categories).astype(str)
    values = item[:]
    return values.astype(str) if values.dtype.kind in 'SO' else values


def lake_cortex_pseudobulk():
    """Raw-count sums of Lake's normal cortex PT nuclei per donor x sex x segment, over all genes (Ensembl ids)."""
    with h5py.File(LAKE, 'r') as handle:
        obs = pd.DataFrame({c: np.asarray(read_obs_column(handle['obs'], c)).astype(str)
                            for c in ['subclass.l2', 'disease', 'region', 'tissue', 'sex', 'donor_id']})
        var_ids = handle['raw/var/_index'][:].astype(str)
        var_names = np.asarray(read_obs_column(handle['raw/var'], 'feature_name')).astype(str)
        keep = (obs['subclass.l2'].isin(['PT-S1', 'PT-S2', 'PT-S3']) & obs.disease.eq('normal')
                & obs.region.eq('C') & obs.tissue.eq('cortex of kidney'))
        label = (obs.donor_id + '|' + obs.sex + '|' + obs['subclass.l2'].str.replace('PT-', '', regex=False))[keep]
        groups_, codes = np.unique(label.to_numpy(), return_inverse=True)
        raw = handle['raw/X']
        sums, totals, nuclei = csr_group_sums(raw['data'], raw['indices'], raw['indptr'], np.flatnonzero(keep.to_numpy()),
                                              codes, len(groups_), len(var_ids), block_rows=5000)
    return {'groups': groups_.astype(str), 'var_ids': var_ids, 'var_names': var_names, 'sums': sums, 'totals': totals,
            'nuclei': nuclei}


t0 = time.perf_counter()
lake = cached('lake_cortex_pt', lake_cortex_pseudobulk,
              params={'labels': 'PT-S1,PT-S2,PT-S3', 'disease': 'normal', 'region': 'C, cortex of kidney',
                      'group': 'donor|sex|segment'},
              inputs={'lake_file': [LAKE.name, LAKE.stat().st_size]})
lake_groups = pd.DataFrame([g.split('|') for g in lake['groups'].astype(str)], columns=['donor', 'sex', 'segment'])
lake_groups['nuclei'] = lake['nuclei']
assert not lake_groups.duplicated(['donor', 'segment']).any(), 'a Lake donor carries two sexes'

# Lake on our genes: Ensembl id first, then symbol (notebook 38's rule); duplicated targets are summed.
handle = ad.read_h5ad(MATRICES['HUK1_COR1'], backed='r')
human_panel_ids = handle.var['gene_ids'].astype(str)
handle.file.close()
by_id = dict(zip(orthologs.human_symbol.map(human_panel_ids), orthologs.mouse_symbol))
by_symbol = dict(zip(orthologs.human_symbol, orthologs.mouse_symbol))
target = np.array([by_id.get(i, by_symbol.get(s, '')) for i, s in zip(lake['var_ids'].astype(str), lake['var_names'].astype(str))])
lake_mapped = target != ''
lake_on_genes = (pd.DataFrame(lake['sums'][:, lake_mapped], columns=target[lake_mapped]).T.groupby(level=0).sum().T
                 .reindex(columns=GENES, fill_value=0.0))
INFO['lake_mapped'] = GENES.isin(set(target[lake_mapped]))
print(f"Lake: {len(lake_groups)} cortex donor-segment groups · {int(lake_mapped.sum()):,} Lake genes mapped · "
      f"{int(INFO.lake_mapped.sum()):,} of our genes mapped ({time.perf_counter() - t0:.0f} s)")

In [ ]:
# Guards (not cached): notebook 62's all-gene cortex cache and notebook 38's cache on our genes.
stored62 = newest_payload(nb62 / 'stage_cache', 'lake_cortex_pt_all_genes')
if stored62 is None:
    guard_log.append("notebook 62's Lake cortex cache is absent: its guard was skipped")
else:
    row = {f'{d}|{s}': k for k, (d, s) in enumerate(zip(lake_groups.donor, lake_groups.segment))}
    names62 = stored62['groups'].astype(str)
    assert sorted(row) == sorted(names62) and list(stored62['genes'].astype(str)) == list(lake['var_ids'].astype(str))
    order = [row[g] for g in names62]
    np.testing.assert_array_equal(lake['sums'][order], stored62['sums'])
    np.testing.assert_array_equal(lake['totals'][order], stored62['totals'])
    np.testing.assert_array_equal(lake['nuclei'][order], stored62['counts'])
    guard_log.append("Lake cortex pseudobulk equals notebook 62's cache (all genes)")
stored38 = newest_payload(nb38 / 'stage_cache', 'lake_pt_pseudobulk')
if stored38 is None:
    guard_log.append("notebook 38's Lake cache is absent: its guard was skipped")
else:
    assert list(stored38['genes'].astype(str)) == list(GENES)
    names38 = pd.Index(stored38['groups'].astype(str))
    ours38 = (lake_groups.donor + '|' + lake_groups.sex + '|' + lake_groups.segment + '|cortex').tolist()
    rows38 = names38.get_indexer(ours38)
    assert (rows38 >= 0).all(), 'a cortex group is missing from notebook 38'
    np.testing.assert_allclose(lake_on_genes.to_numpy(), stored38['sums'][rows38], rtol=0, atol=1e-6)
    guard_log.append("Lake on our genes equals notebook 38's cache for the cortex groups")
del stored62, stored38
print('\n'.join(guard_log[-2:]))

In [ ]:
# Donors and per-donor x segment sums (units x segments x genes)
nuclei = lake_groups.pivot(index='donor', columns='segment', values='nuclei').reindex(columns=list(SEGMENTS)).fillna(0)
lake_donors = sorted(nuclei.index[(nuclei >= MIN_NUCLEI).all(axis=1)])
assert lake_donors == sorted(LAKE_DONORS), lake_donors
lake_sex = lake_groups.drop_duplicates('donor').set_index('donor').sex
lake_row = {(d, s): k for k, (d, s) in enumerate(zip(lake_groups.donor, lake_groups.segment))}


def lake_block(donors, matrix):
    return np.stack([np.stack([np.asarray(matrix[lake_row[(d, s)]], float) for s in SEGMENTS]) for d in donors])


LAKE_SUMS = lake_block(lake_donors, lake_on_genes.to_numpy())          # our genes
LAKE_ENSEMBL = lake_block(lake_donors, lake['sums'])                    # every Lake gene (families)
LAKE_IDS = pd.Index(lake['var_ids'].astype(str))
lake_male = np.array([lake_sex[d] == 'male' for d in lake_donors])


def census_table(label, suffix=''):
    table = pd.read_csv(CENSUS / f'{label}_pt_segment_pseudobulk_counts{suffix}.csv.gz', index_col=0)
    meta_ = pd.read_csv(CENSUS / f'{label}_pt_segment_pseudobulk_meta{suffix}.csv', index_col=0)
    return table, meta_


def census_donors(meta_, segment_map, *, sex=None, stages=None):
    """Census groups of donors with >= MIN_NUCLEI nuclei in every mapped segment (embryonic stages removed)."""
    m = meta_[~meta_.development_stage.str.contains('embryo|E16|Theiler', case=False)].copy()
    m['segment'] = m.cell_type.map(segment_map)
    m = m[m.segment.notna() & m.n_cells.ge(MIN_NUCLEI)]
    if sex is not None:
        m = m[m.sex.eq(sex)]
    if stages is not None:
        m = m[m.development_stage.isin(stages)]
    complete = m.groupby('donor_id').segment.apply(lambda s: set(s) == set(segment_map.values()))
    return m[m.donor_id.isin(complete[complete].index)]


def census_block(table, m, donors, segments):
    group_of = pd.Series(m.index, index=pd.MultiIndex.from_arrays([m.donor_id, m.segment]))
    assert group_of.index.is_unique
    return np.stack([table.loc[[group_of[(d, s)] for s in segments]].to_numpy(float) for d in donors])


mouse_counts, mouse_meta = census_table('mouse')
human_counts_c, human_meta_c = census_table('human')
family_counts, family_meta = census_table('mouse', '_families')
mouse_adult = census_donors(mouse_meta, CENSUS_SEGMENTS, sex='male', stages=ADULT_STAGES)
mouse_males = census_donors(mouse_meta, CENSUS_SEGMENTS, sex='male')
census_human = census_donors(human_meta_c, CENSUS_HUMAN_SEGMENTS)
ADULT, MALES, CENSUS_HUMAN = (sorted(m.donor_id.unique()) for m in (mouse_adult, mouse_males, census_human))
assert (len(ADULT), len(MALES), len(CENSUS_HUMAN)) == (N_ADULT_MALES, N_MALES, N_CENSUS_HUMAN), (ADULT, MALES, CENSUS_HUMAN)

# Guards (not cached): the family fetch has the main file's groups, nuclei and totals; shared columns are identical
# except where the symbol fetch measured another gene (audited other_id).
pd.testing.assert_frame_equal(family_meta.loc[mouse_meta.index], mouse_meta)
shared = sorted(set(family_counts.columns) & set(mouse_counts.columns))
differ = [g for g in shared if not np.array_equal(family_counts.loc[mouse_meta.index, g].to_numpy(),
                                                  mouse_counts.loc[mouse_meta.index, g].to_numpy())]
assert set(differ) <= census_other['mouse'], differ
guard_log.append(f'Census family fetch: groups and totals equal the main file; {len(shared)} shared genes identical except '
                 f'{differ} (audited: the symbol fetch measured another gene)')
print(guard_log[-1])

mouse_on_genes = mouse_counts.reindex(columns=GENES).fillna(0.0)
human_c_on_genes = human_counts_c.reindex(columns=GENES).fillna(0.0)
MOUSE_ADULT_SUMS = census_block(mouse_on_genes, mouse_adult, ADULT, SEGMENTS)
MOUSE_MALE_SUMS = census_block(mouse_on_genes, mouse_males, MALES, SEGMENTS)
CENSUS_HUMAN_SUMS = census_block(human_c_on_genes, census_human, CENSUS_HUMAN, ('PCT', 'S3'))
INFO['in_census_mouse'] = GENES.isin(mouse_counts.columns) & ~GENES.isin(census_other['mouse'])
INFO['in_census_human'] = GENES.isin(human_counts_c.columns) & ~GENES.isin(census_other['human'])
ANALYSED = (INFO.lake_mapped & INFO.in_census_mouse).to_numpy()
ANALYSED_CENSUS_HUMAN = (INFO.in_census_human & INFO.in_census_mouse).to_numpy()
DONOR_TABLE = pd.DataFrame([
    {'atlas': 'Lake (human, primary)', 'donors': len(lake_donors), 'male': int(lake_male.sum()), 'ids': ', '.join(lake_donors)},
    {'atlas': 'Census mouse adult males (primary)', 'donors': len(ADULT), 'male': len(ADULT), 'ids': ', '.join(ADULT)},
    {'atlas': 'Census mouse, all males', 'donors': len(MALES), 'male': len(MALES), 'ids': ', '.join(MALES)},
    {'atlas': 'Census human cortex', 'donors': len(CENSUS_HUMAN),
     'male': int(census_human.drop_duplicates('donor_id').sex.eq('male').sum()), 'ids': ', '.join(CENSUS_HUMAN)}])
DONOR_TABLE.to_csv(tables / 'atlas_donors.csv', index=False)
display(DONOR_TABLE)
print(f'Analysed genes: {int(ANALYSED.sum()):,} (Lake vs Census mouse) · {int(ANALYSED_CENSUS_HUMAN.sum()):,} '
      '(Census human vs Census mouse)')

In [ ]:
def atlas_comparison(human_sums, mouse_sums, mask, *, human_weights=None):
    """limma-trend human vs mouse on donor-level equal-segment log-CPM over the analysed genes ``mask``."""
    idx = np.flatnonzero(mask)
    h, m = human_sums[:, :, idx], mouse_sums[:, :, idx]
    totals_h, totals_m = h.sum(axis=2), m.sum(axis=2)
    values = np.vstack([equal_segment_log_cpm(h, totals_h, weights=human_weights), equal_segment_log_cpm(m, totals_m)])
    case = np.r_[np.ones(len(h), bool), np.zeros(len(m), bool)]
    fit = limma_trend_species(values, case, genes=GENES[idx])
    return {'fit': fit.reindex(GENES), 'human_totals': totals_h, 'mouse_totals': totals_m,
            'centre': float(np.median(fit.coefficient))}


t0 = time.perf_counter()
ATLAS = {
    'primary': atlas_comparison(LAKE_SUMS, MOUSE_ADULT_SUMS, ANALYSED),
    'all males': atlas_comparison(LAKE_SUMS, MOUSE_MALE_SUMS, ANALYSED),
    'Lake men': atlas_comparison(LAKE_SUMS[lake_male], MOUSE_ADULT_SUMS, ANALYSED),
    'Census human': atlas_comparison(CENSUS_HUMAN_SUMS, MOUSE_ADULT_SUMS, ANALYSED_CENSUS_HUMAN,
                                     human_weights=CENSUS_HUMAN_WEIGHTS)}
A = ATLAS['primary']['fit'].centred.to_numpy()
A_Q = ATLAS['primary']['fit'].q.to_numpy()


def atlas_strong(fit):
    a, q = fit.centred.to_numpy(), fit.q.to_numpy()
    return np.isfinite(a) & (q <= ATLAS_FDR) & (np.abs(a) >= ATLAS_LFC)


STRONG = atlas_strong(ATLAS['primary']['fit'])
ATLAS_TABLE = pd.DataFrame([{
    'atlas': name, 'genes': int(item['fit'].centred.notna().sum()), 'centre (log2)': item['centre'],
    'atlas-strong': int(atlas_strong(item['fit']).sum()),
    'median prior df': float(np.nanmedian(item['fit'].df_prior))} for name, item in ATLAS.items()])
ATLAS_TABLE.to_csv(tables / 'atlas_fits.csv', index=False)
pd.concat({name: item['fit'][['coefficient', 'centred', 'se', 't_centred', 'p', 'q', 'amean']]
           for name, item in ATLAS.items()}, axis=1).to_csv(tables / 'atlas_coefficients.csv')
display(ATLAS_TABLE)
print(f'Atlas fits {time.perf_counter() - t0:.0f} s')

## 4 · Our level

- **L_g** = (1 / ln 2) · mean over the grid of [Δ_g(x) − m(x)], with m(x) the median of Δ at x over the
  reference genes (converged and not separated). Non-converged genes are dropped.
- **Pair spread.** The same statistic from the specimen curves for the four human-section × mouse pairs, each
  centred by its own median. A pair is missing where either specimen fit is separated or did not converge.
- **Same-species reference.** Ctrl1A2 − Ctrl1A4 and HUK1_COR1 − HUK1_MED1, by the same formula.
- **P_g.**
  - Per specimen × segment pseudobulk log-CPM, log2((y + 0.5) / (N + 1) · 10⁶), with N the summed library.
  - The equal-weight mean over S1, S2 and S3.
  - Human mean minus mouse mean, minus the median over the reference genes.
- **Direction used in sign checks.** L, except for separated genes, which use P (and |P| for |L|).

In [ ]:
stats = MODEL.statistics
CONVERGED = stats.converged.to_numpy()
SEPARATED = stats.separated.to_numpy()
REF = CONVERGED & ~SEPARATED
L = grid_centred_level(pd.DataFrame(MODEL.delta, index=GENES), reference=REF).to_numpy()
L[~CONVERGED] = np.nan

valid = {n: (MODEL.specimen_converged[n] & ~MODEL.specimen_separated[n]).to_numpy() & CONVERGED for n in MOUSE + HUMAN}
CROSS_PAIRS = [(h, m) for h in HUMAN for m in MOUSE]
PAIRS = pair_levels(MODEL.specimen_curves, CROSS_PAIRS, valid=valid, reference=REF, index=GENES)
PAIR_SUMMARY = pair_summary(PAIRS)
SAME = pair_levels(MODEL.specimen_curves, [tuple(MOUSE), tuple(HUMAN)], valid=valid, reference=REF, index=GENES)

UNITS, P_SUMS, P_TOTALS = segment_pseudobulk(counts, library, specimen, segment, segments=SEGMENTS)
CASE_UNITS = np.isin(UNITS, HUMAN)
P_RAW = species_difference(equal_segment_log_cpm(P_SUMS, P_TOTALS), CASE_UNITS)
P_CENTRE = float(np.median(P_RAW[REF]))
P = P_RAW - P_CENTRE
P[~CONVERGED] = np.nan
DIRECTION = np.where(SEPARATED, P, L)
MAGNITUDE = np.abs(DIRECTION)

GENE_TABLE = pd.concat([pd.DataFrame({
    'L': L, 'P': P, 'direction': DIRECTION, 'converged': CONVERGED, 'separated': SEPARATED, 'reference': REF,
    'T_spatial': stats.T_spatial.to_numpy(), 'A': A, 'A_q': A_Q, 'atlas_strong': STRONG,
    'A_all_males': ATLAS['all males']['fit'].centred.to_numpy(), 'A_lake_men': ATLAS['Lake men']['fit'].centred.to_numpy(),
    'A_census_human': ATLAS['Census human']['fit'].centred.to_numpy()}, index=GENES),
    PAIRS, PAIR_SUMMARY, SAME.add_prefix('same species: '), INFO], axis=1)
GENE_TABLE.to_csv(tables / 'gene_level.csv')
bridge = bootstrap_spearman(L[REF], P[REF], n_boot=N_BOOT, seed=SEED)
print(f'L: {int(REF.sum()):,} reference genes · Spearman L vs P {bridge["rho"]:.3f} '
      f'[{bridge["low"]:.3f}, {bridge["high"]:.3f}] · all four pairs share the sign for '
      f'{int(PAIR_SUMMARY.pairs_same_sign[REF].sum()):,} reference genes')

## 5 · Bands: the housekeeping yardstick

**The housekeeping band.**
- The list: Eisenberg & Levanon 2013, mapped by human symbol through the ortholog map.
- The band: the 2.5–97.5% quantiles of L over the housekeeping genes that are reference genes.
- A control clears it when it lies beyond the band on its expected side.

**Other bands.** The same band is computed for P (families and separated genes) and for A (the atlas). The
width of the A band is what levels spread to among housekeeping genes without probes. The same-species bands
show how far two mice, or two sections, differ by this formula.

In [ ]:
HK = INFO.housekeeping.to_numpy()
BANDS = {
    'L, housekeeping': (L, HK & REF), 'P, housekeeping': (P, HK & REF),
    'A, housekeeping': (A, HK & ANALYSED & np.isfinite(A)),
    'mouse − mouse, all genes': (SAME[f'{MOUSE[0]} - {MOUSE[1]}'].to_numpy(), REF),
    'section − section, all genes': (SAME[f'{HUMAN[0]} - {HUMAN[1]}'].to_numpy(), REF),
    'L, all genes': (L, REF)}
BAND_TABLE = pd.DataFrame([{'band': name, 'genes': int((mask & np.isfinite(values)).sum()),
                            'low': band(values[mask], BAND_QUANTILES)[0], 'high': band(values[mask], BAND_QUANTILES)[1]}
                           for name, (values, mask) in BANDS.items()]).set_index('band')
BAND_TABLE['width'] = BAND_TABLE.high - BAND_TABLE.low
BAND_TABLE.to_csv(tables / 'bands.csv')
HK_BAND = {'L': tuple(BAND_TABLE.loc['L, housekeeping', ['low', 'high']]),
           'P': tuple(BAND_TABLE.loc['P, housekeeping', ['low', 'high']])}
display(BAND_TABLE.round(3))
print(f"Housekeeping band width, ours (L) / atlas (A): "
      f"{BAND_TABLE.loc['L, housekeeping', 'width'] / BAND_TABLE.loc['A, housekeeping', 'width']:.2f}")

## 6 · Literature controls: criterion (i)

The controls are `docs/results/pt-level-literature-controls.csv` (34 rows). They were compiled blind to our data,
are frozen by sha256 and are used as written.

**What they measure.** 29 of the 30 rows verified for mouse come from one study: Thakur et al. 2024 (PMID
38711199), LC-MS/MS of membrane transporters and enzymes in kidney membrane fractions.
- These are **protein** abundances.
- The mouse samples are whole kidney and the human samples tissue sections, pooled over sex.
- So a sign disagreement can come from mRNA–protein discordance or from the tissue, not only from our
  measurement.

**Routing.**
- A gene row is evaluated when its symbol is one of our genes: the mouse symbol directly, else the human
  symbol through the ortholog map.
- A gene outside our universe that sits in a non-1:1 orthogroup goes to the family check (section 8).
- Pathway rows (L16, L17) go to section 9.

**Criterion (i).** Only gene rows with `direction_verified_for_mouse = yes` count. A control is correct when
sign(L) equals the expected direction (sign(P) for separated genes). It needs ≥ 80% correct, with ≥ 10
evaluable controls.

Reported beside it:
- the Wilson interval;
- the controls clearing the housekeeping band (the P band for separated genes);
- the controls whose four pairs all share the expected sign;
- the equal-probe, not-nearly-absent and not-separated subsets;
- the atlas sign of each control.

The rows not verified for mouse (L02, L07, L14) are reported only.

In [ ]:
lit = pd.read_csv(LITERATURE)
LITERATURE_COLUMNS = ['control_id', 'kind', 'human_genes', 'mouse_genes', 'expected', 'tissue', 'measurement',
                      'confidence', 'direction_verified_for_mouse', 'citation']
assert set(LITERATURE_COLUMNS) <= set(lit.columns) and lit.control_id.is_unique, 'the literature file changed'
lit['expected_sign'] = lit.expected.str.strip().str.lower().map(EXPECTED_SIGN)
if lit.expected_sign.isna().any():
    raise ValueError(f'Unknown expected directions; the frozen mapping is {EXPECTED_SIGN}.')
lit['verified_for_mouse'] = lit.direction_verified_for_mouse.str.strip().str.lower().eq('yes')
lit['kind'] = lit.kind.str.strip().str.lower()
assert set(lit.kind) <= {'gene', 'pathway'}

component_of = {sp: inventory[sp][inventory[sp].ortholog_class.isin(FAMILY_CLASSES)]
                .assign(key=lambda t: t.gene.str.upper()).set_index('key').component for sp in ('mouse', 'human')}


def our_gene(mouse_symbol, human_symbol_):
    """Our gene for a control: the mouse symbol if it is one of our genes, else the human symbol's ortholog."""
    if mouse_symbol in GENES:
        return mouse_symbol
    mouse = to_mouse.get(str(human_symbol_).upper())
    return mouse if mouse in GENES else None


def route(row):
    """(route, key): gene -> our mouse symbol; family -> orthogroup; pathway -> member genes; else not evaluable."""
    if row.kind == 'pathway':
        return 'pathway', None
    gene = our_gene(str(row.mouse_genes).strip(), str(row.human_genes).strip())
    if gene is not None:
        return 'gene', gene
    for sp, symbol in (('mouse', row.mouse_genes), ('human', row.human_genes)):
        component = component_of[sp].get(str(symbol).strip().upper())
        if component is not None:
            return 'family', int(component)
    return 'not evaluable', None


routes = [route(row) for row in lit.itertuples()]
lit['route'] = [r for r, _ in routes]
lit['key'] = [k for _, k in routes]
gene_rows = lit[lit.route.eq('gene')].copy()
position_of = pd.Series(np.arange(len(GENES)), index=GENES)
k = position_of.reindex(gene_rows.key).to_numpy().astype(int)
gene_rows['separated'] = SEPARATED[k]
gene_rows['converged'] = CONVERGED[k]
gene_rows['L'] = L[k]
gene_rows['P'] = P[k]
gene_rows['value'] = DIRECTION[k]
gene_rows['correct'] = np.sign(gene_rows.value) == gene_rows.expected_sign
gene_rows['clears_band'] = np.where(gene_rows.separated, clears_band(gene_rows.value, gene_rows.expected_sign, HK_BAND['P']),
                                    clears_band(gene_rows.value, gene_rows.expected_sign, HK_BAND['L']))
gene_rows['pairs_same_sign_as_expected'] = (PAIR_SUMMARY.pairs_same_sign.to_numpy()[k]
                                            & (np.sign(PAIR_SUMMARY.pair_min.to_numpy()[k]) == gene_rows.expected_sign))
for column in ('pair_min', 'pair_max'):
    gene_rows[column] = PAIR_SUMMARY[column].to_numpy()[k]
for column in ('equal_probes', 'probe_clean_both', 'nearly_absent', 'mouse_probes', 'human_probes'):
    gene_rows[column] = INFO[column].to_numpy()[k]
gene_rows['A'] = A[k]
gene_rows['A_q'] = A_Q[k]
gene_rows['atlas_sign_as_expected'] = np.sign(gene_rows.A) == gene_rows.expected_sign
gene_rows.to_csv(tables / 'literature_genes.csv', index=False)
lit.to_csv(tables / 'literature_routing.csv', index=False)


def criterion_share(correct, minimum):
    n, hit = int(len(correct)), int(np.sum(correct))
    share = hit / n if n else np.nan
    low, high = wilson_interval(hit, n)
    status = 'not evaluable' if n < minimum else ('pass' if share >= PASS_SHARE else 'fail')
    return {'n': n, 'agree': hit, 'share': share, 'wilson_low': low, 'wilson_high': high, 'status': status}


usable = gene_rows[gene_rows.converged & gene_rows.value.notna()]
evaluable = usable[usable.verified_for_mouse]
subsets = {'primary (criterion i)': evaluable,
           'primary, equal probes': evaluable[evaluable.equal_probes],
           'primary, equal probes, probe-clean in both': evaluable[evaluable.equal_probes & evaluable.probe_clean_both],
           'primary, not nearly absent': evaluable[~evaluable.nearly_absent],
           'primary, not separated': evaluable[~evaluable.separated],
           'not verified for mouse (reported only)': usable[~usable.verified_for_mouse]}
LIT_TABLE = pd.DataFrame({name: criterion_share(rows.correct, MIN_LITERATURE) for name, rows in subsets.items()}).T
LIT_TABLE['clear the housekeeping band'] = [int(rows.clears_band.sum()) for rows in subsets.values()]
LIT_TABLE['four pairs share the expected sign'] = [int(rows.pairs_same_sign_as_expected.sum()) for rows in subsets.values()]
LIT_TABLE['atlas A has the expected sign'] = [int(rows.atlas_sign_as_expected.sum()) for rows in subsets.values()]
LIT_TABLE.loc['not verified for mouse (reported only)', 'status'] = 'reported only'
CRITERION_I = LIT_TABLE.loc['primary (criterion i)', ['n', 'agree', 'share', 'wilson_low', 'wilson_high', 'status']].to_dict()
LIT_TABLE.to_csv(tables / 'criterion_i.csv')
display(lit.groupby(['kind', 'route', 'verified_for_mouse']).size().rename('rows').to_frame())
display(LIT_TABLE)
display(gene_rows.sort_values('value')[['control_id', 'key', 'expected', 'verified_for_mouse', 'value', 'correct',
                                        'clears_band', 'pair_min', 'pair_max', 'separated', 'equal_probes',
                                        'mouse_probes', 'human_probes', 'A', 'A_q']].round(2))

## 7 · Atlas agreement: criteria (ii) and (iii)

**(ii)** Among genes that are atlas-strong (BH ≤ 0.05, |A| ≥ 1) and have |L| ≥ 1, ≥ 80% agree in sign with A. It
needs ≥ 30 genes. Separated genes enter with P.

Reported beside it:
- all atlas-strong genes regardless of |L|;
- P in place of L;
- the equal-probe genes, also requiring probe-clean in both species;
- without nearly absent or separated genes;
- without the pairs inside non-1:1 orthogroups (notebook 64);
- the atlas sensitivities: all 12 males, Lake's men, Census human.

**(iii)** The Spearman correlation of L with A, with a 2,000-gene bootstrap interval (seed 68), on the reference
genes that are analysed in the atlas. It is set against:
- the equal-probe genes;
- P;
- the atlas ceiling, A against Census human's A over the same mice;
- the probe-ratio diagnostic: the Spearman correlation of L − A with log2(human probes / mouse probes) over genes
  with unequal probe counts.

In [ ]:
def agreement(direction, magnitude, fit, mask, level_min=LEVEL_MIN):
    a = fit.centred.to_numpy()
    use = atlas_strong(fit) & mask & np.isfinite(direction) & (magnitude >= level_min)
    return criterion_share(np.sign(direction[use]) == np.sign(a[use]), MIN_ATLAS_GENES)


EQUAL = INFO.equal_probes.to_numpy()
CLEAN_BOTH = INFO.probe_clean_both.to_numpy()
ABSENT_ = INFO.nearly_absent.to_numpy()
PAIRED_IN_FAMILY = INFO.non_one_to_one_pair.to_numpy()
primary_fit = ATLAS['primary']['fit']
AGREEMENT = pd.DataFrame({
    'primary (criterion ii)': agreement(DIRECTION, MAGNITUDE, primary_fit, CONVERGED),
    'all atlas-strong genes, any |L|': agreement(DIRECTION, MAGNITUDE, primary_fit, CONVERGED, level_min=0),
    'P in place of L': agreement(P, np.abs(P), primary_fit, CONVERGED),
    'equal probes': agreement(DIRECTION, MAGNITUDE, primary_fit, CONVERGED & EQUAL),
    'equal probes, probe-clean in both': agreement(DIRECTION, MAGNITUDE, primary_fit, CONVERGED & EQUAL & CLEAN_BOTH),
    'without nearly absent genes': agreement(DIRECTION, MAGNITUDE, primary_fit, CONVERGED & ~ABSENT_),
    'without separated genes': agreement(DIRECTION, MAGNITUDE, primary_fit, REF),
    'without pairs in non-1:1 orthogroups': agreement(DIRECTION, MAGNITUDE, primary_fit, CONVERGED & ~PAIRED_IN_FAMILY),
    'atlas: all 12 Census males': agreement(DIRECTION, MAGNITUDE, ATLAS['all males']['fit'], CONVERGED),
    "atlas: Lake's 3 men": agreement(DIRECTION, MAGNITUDE, ATLAS['Lake men']['fit'], CONVERGED),
    'atlas: Census human for Lake': agreement(DIRECTION, MAGNITUDE, ATLAS['Census human']['fit'], CONVERGED)}).T
CRITERION_II = AGREEMENT.loc['primary (criterion ii)'].to_dict()
AGREEMENT.to_csv(tables / 'criterion_ii.csv')
display(AGREEMENT)

with np.errstate(divide='ignore', invalid='ignore'):
    probe_ratio = np.log2(INFO.human_probes.to_numpy(float) / INFO.mouse_probes.to_numpy(float))
both_atlas = ANALYSED & ANALYSED_CENSUS_HUMAN
SPEARMAN_ROWS = {
    'L vs A (criterion iii)': (L, A, REF & ANALYSED),
    'L vs A, equal probes': (L, A, REF & ANALYSED & EQUAL),
    'L vs A, equal probes and probe-clean in both': (L, A, REF & ANALYSED & EQUAL & CLEAN_BOTH),
    'L vs A, without nearly absent genes': (L, A, REF & ANALYSED & ~ABSENT_),
    'L vs A, without pairs in non-1:1 orthogroups': (L, A, REF & ANALYSED & ~PAIRED_IN_FAMILY),
    'P vs A': (P, A, REF & ANALYSED),
    'L vs A (all 12 Census males)': (L, ATLAS['all males']['fit'].centred.to_numpy(), REF & ANALYSED),
    'L vs A (Census human)': (L, ATLAS['Census human']['fit'].centred.to_numpy(), REF & ANALYSED_CENSUS_HUMAN),
    'atlas ceiling: A (Lake) vs A (Census human)': (A, ATLAS['Census human']['fit'].centred.to_numpy(), both_atlas),
    'L vs P': (L, P, REF),
    'probe diagnostic: L - A vs log2(human / mouse probes), unequal probes':
        (L - A, probe_ratio, REF & ANALYSED & ~EQUAL & np.isfinite(probe_ratio))}
SPEARMAN = pd.DataFrame({name: bootstrap_spearman(x[mask], y[mask], n_boot=N_BOOT, seed=SEED)
                         for name, (x, y, mask) in SPEARMAN_ROWS.items()}).T
SPEARMAN.to_csv(tables / 'criterion_iii_spearman.csv')
display(SPEARMAN)

## 8 · Families: genome-defined unequal member counts (iv, descriptive)

**Orthogroups.** Notebook 62's HCOP components (support ≥ 3) of class 1:many, many:1 or many:many, kept when:
- each species has ≥ 1 member on its panel;
- the HCOP member counts are unequal.

The pre-named families are added. Ugt2b has 8 members in each species, so it carries no expectation.

**Family sum.** Per structure, the raw counts of every panel member of the species, summed. Our structures are
read from the native matrices `tubule_by_gene/<sample>`. Summing absorbs within-family cross-hybridisation, so
this is the primary; probe-clean members only is a sensitivity.

**F_f** is P's statistic on the family sum:
- equal-segment log-CPM against the gene model's library;
- human minus mouse;
- minus the median of P over the reference genes.

**Expectation** (explicitly weaker): sign(F) = sign(n_human − n_mouse), with n the HCOP member counts.
- It is evaluated on the families detected in ≥ 2% of the structures of both specimens of at least one species.
- It is reported separately for the pre-named families.

**Atlas F.** The same statistic in Lake (Ensembl ids) and in the Census adult males, against the atlas
libraries and centred by the median atlas coefficient. It is computed only for families with every member found
in both atlases.

In [ ]:
catalogue = pd.read_csv(nb62 / 'tables' / 'family_catalogue.csv')
for column in ('mouse n panel', 'human n panel', 'mouse total HCOP members', 'human total HCOP members'):
    catalogue[column] = catalogue[column].fillna(0).astype(int)
catalogue = catalogue.set_index('component')
catalogue['eligible'] = (catalogue['mouse n panel'].gt(0) & catalogue['human n panel'].gt(0)
                         & catalogue['mouse total HCOP members'].ne(catalogue['human total HCOP members']))
anchor_component = {name: int(inventory['mouse'].set_index('gene').component[anchor]) for name, anchor in PRENAMED.items()}
prenamed_of = {component: name for name, component in anchor_component.items()}
assert set(prenamed_of) <= set(catalogue.index), 'a pre-named family is not a notebook 62 orthogroup'
FAMILY_SET = sorted({int(c) for c in catalogue.index[catalogue.eligible]} | set(prenamed_of))
MEMBERS = {}
for sp in ('mouse', 'human'):
    table = inventory[sp][inventory[sp].ortholog_class.isin(FAMILY_CLASSES) & inventory[sp].component.isin(FAMILY_SET)]
    sizes = table.groupby('component').size().reindex(FAMILY_SET).fillna(0).astype(int)
    assert (sizes.to_numpy() == catalogue.loc[FAMILY_SET, f'{sp} n panel'].to_numpy()).all(), f'{sp} members changed'
    for variant, rows in (('all', table), ('clean', table[table.probe_clean.astype(bool)])):
        MEMBERS[(sp, variant)] = {int(c): sorted(g.gene_id.astype(str)) for c, g in rows.groupby('component')}
        MEMBERS[(sp, variant, 'symbols')] = {int(c): sorted(g.gene.astype(str)) for c, g in rows.groupby('component')}
print(f'{len(FAMILY_SET)} orthogroups ({int(catalogue.eligible.sum())} eligible + pre-named); members '
      + ', '.join(f"{sp}: {sum(map(len, MEMBERS[(sp, 'all')].values())):,}" for sp in ('mouse', 'human')))


def visium_family_sums():
    """Per specimen x segment family sums of raw counts from the native panels, and per-specimen detection."""
    out = {}
    for variant in ('all', 'clean'):
        out[f'sums_{variant}'] = np.zeros((len(UNITS), len(SEGMENTS), len(FAMILY_SET)))
        out[f'detected_{variant}'] = np.zeros((len(UNITS), len(FAMILY_SET)))
        out[f'available_{variant}'] = np.zeros((len(UNITS), len(FAMILY_SET)), bool)
    for u, name in enumerate(UNITS):
        species_ = 'human' if name in HUMAN else 'mouse'
        native = ad.read_h5ad(MATRICES[name])
        rows = pd.Index(structure_ids[specimen == name])
        if not rows.str.startswith(name + '_').all():
            raise ValueError(f'{name}: structure ids lack their specimen prefix.')
        local = rows.str.slice(len(name) + 1)
        absent = local.difference(native.obs_names)
        if len(absent):
            raise ValueError(f'{name}: {len(absent)} structures are missing from the native matrix.')
        x = sparse.csr_matrix(native[local].X, dtype=float)
        if (x.data < 0).any() or not np.allclose(x.data, np.round(x.data)):
            raise ValueError(f'{name}: expected raw integer counts.')
        ids = native.var['gene_ids'].astype(str).to_numpy()
        seg = segment[specimen == name]
        for variant in ('all', 'clean'):
            members = MEMBERS[(species_, variant)]
            present = [c for c in FAMILY_SET if members.get(c)]
            sums, _ = family_sums(x, ids, {c: members[c] for c in present})
            columns = [FAMILY_SET.index(c) for c in present]
            for j, s in enumerate(SEGMENTS):
                out[f'sums_{variant}'][u, j, columns] = np.asarray(sums[seg == s].sum(axis=0)).ravel()
            out[f'detected_{variant}'][u, columns] = np.asarray((sums > 0).mean(axis=0)).ravel()
            out[f'available_{variant}'][u, columns] = True
    return out


t0 = time.perf_counter()
FAMILY_PAYLOAD = cached('visium_family_sums', visium_family_sums,
                        params={'segments': SEGMENTS, 'classes': FAMILY_CLASSES},
                        inputs={'structures': structure_ids, 'segment': segment, 'specimen': specimen,
                                'members': {f'{k[0]}|{k[1]}': {str(c): m for c, m in v.items()}
                                            for k, v in MEMBERS.items() if len(k) == 2},
                                'matrices': {s: [p.name, p.stat().st_size] for s, p in MATRICES.items()}})
print(f'Visium family sums {time.perf_counter() - t0:.0f} s')

In [ ]:
F, DETECTED = {}, {}
for variant in ('all', 'clean'):
    values = equal_segment_log_cpm(FAMILY_PAYLOAD[f'sums_{variant}'], P_TOTALS)
    f = species_difference(values, CASE_UNITS) - P_CENTRE
    f[~FAMILY_PAYLOAD[f'available_{variant}'].all(axis=0)] = np.nan
    F[variant] = f
    detected = FAMILY_PAYLOAD[f'detected_{variant}'] >= FAMILY_DETECTION
    DETECTED[variant] = detected[CASE_UNITS].all(axis=0) | detected[~CASE_UNITS].all(axis=0)

# Atlas family sums: Lake by Ensembl id (human members), Census adult males by our symbol (mouse members).
fetched = pd.read_csv(CENSUS / 'mouse_family_genes_fetched.csv').set_index('gene').found.astype(bool)
mouse_family_block = census_block(family_counts, mouse_adult, ADULT, SEGMENTS)
family_columns = pd.Index(family_counts.columns)
atlas_human = np.full((len(lake_donors), len(SEGMENTS), len(FAMILY_SET)), np.nan)
atlas_mouse = np.full((len(ADULT), len(SEGMENTS), len(FAMILY_SET)), np.nan)
for j, c in enumerate(FAMILY_SET):
    h_idx = LAKE_IDS.get_indexer(MEMBERS[('human', 'all')].get(c, []))
    m_symbols = MEMBERS[('mouse', 'all', 'symbols')].get(c, [])
    m_idx = family_columns.get_indexer(m_symbols)
    if len(h_idx) and (h_idx >= 0).all():
        atlas_human[:, :, j] = LAKE_ENSEMBL[:, :, h_idx].sum(axis=2)
    if len(m_idx) and (m_idx >= 0).all() and fetched.reindex(m_symbols).fillna(False).all():
        atlas_mouse[:, :, j] = mouse_family_block[:, :, m_idx].sum(axis=2)
complete = np.isfinite(atlas_human).all(axis=(0, 1)) & np.isfinite(atlas_mouse).all(axis=(0, 1))
ATLAS_F = np.full(len(FAMILY_SET), np.nan)
if complete.any():
    h_values = equal_segment_log_cpm(atlas_human[:, :, complete], ATLAS['primary']['human_totals'])
    m_values = equal_segment_log_cpm(atlas_mouse[:, :, complete], ATLAS['primary']['mouse_totals'])
    ATLAS_F[complete] = h_values.mean(axis=0) - m_values.mean(axis=0) - ATLAS['primary']['centre']

FAMILY_TABLE = pd.DataFrame({
    'family': [prenamed_of.get(c, '') for c in FAMILY_SET],
    'mouse members': [', '.join(MEMBERS[('mouse', 'all', 'symbols')].get(c, [])) for c in FAMILY_SET],
    'human members': [', '.join(MEMBERS[('human', 'all', 'symbols')].get(c, [])) for c in FAMILY_SET],
    'n_mouse_hcop': catalogue.loc[FAMILY_SET, 'mouse total HCOP members'].to_numpy(),
    'n_human_hcop': catalogue.loc[FAMILY_SET, 'human total HCOP members'].to_numpy(),
    'eligible': catalogue.loc[FAMILY_SET, 'eligible'].to_numpy(), 'detected': DETECTED['all'],
    'F': F['all'], 'F_probe_clean': F['clean'], 'atlas_F': ATLAS_F, 'atlas_complete': complete},
    index=pd.Index(FAMILY_SET, name='component'))
FAMILY_TABLE['expected_sign'] = np.sign(FAMILY_TABLE.n_human_hcop - FAMILY_TABLE.n_mouse_hcop)
FAMILY_TABLE['F_as_expected'] = np.sign(FAMILY_TABLE.F) == FAMILY_TABLE.expected_sign
FAMILY_TABLE['F_agrees_with_atlas'] = np.sign(FAMILY_TABLE.F) == np.sign(FAMILY_TABLE.atlas_F)
FAMILY_TABLE['clears_P_band'] = clears_band(FAMILY_TABLE.F, FAMILY_TABLE.expected_sign, HK_BAND['P'])
FAMILY_TABLE.to_csv(tables / 'families.csv')

use = FAMILY_TABLE.eligible & FAMILY_TABLE.detected & FAMILY_TABLE.F.notna()
use_clean = FAMILY_TABLE.eligible & DETECTED['clean'] & FAMILY_TABLE.F_probe_clean.notna()
use_atlas = use & FAMILY_TABLE.atlas_complete
FAMILY_SUMMARY = pd.DataFrame({
    'F in the expected direction': criterion_share(FAMILY_TABLE.F_as_expected[use], 1),
    'F (probe-clean members) in the expected direction': criterion_share(
        (np.sign(FAMILY_TABLE.F_probe_clean) == FAMILY_TABLE.expected_sign)[use_clean], 1),
    'mouse-expanded: F < 0': criterion_share(FAMILY_TABLE.F_as_expected[use & FAMILY_TABLE.expected_sign.lt(0)], 1),
    'human-expanded: F > 0': criterion_share(FAMILY_TABLE.F_as_expected[use & FAMILY_TABLE.expected_sign.gt(0)], 1),
    'atlas F in the expected direction': criterion_share(
        (np.sign(FAMILY_TABLE.atlas_F) == FAMILY_TABLE.expected_sign)[use_atlas], 1),
    'F agrees in sign with atlas F': criterion_share(FAMILY_TABLE.F_agrees_with_atlas[use_atlas], 1)}).T
FAMILY_SUMMARY = FAMILY_SUMMARY.drop(columns='status')
FAMILY_SPEARMAN = bootstrap_spearman(FAMILY_TABLE.F[use_atlas], FAMILY_TABLE.atlas_F[use_atlas], n_boot=N_BOOT, seed=SEED)
FAMILY_SUMMARY.to_csv(tables / 'families_summary.csv')
display(FAMILY_SUMMARY)
print(f"Spearman F vs atlas F over {FAMILY_SPEARMAN['n']} complete families: {FAMILY_SPEARMAN['rho']:.2f} "
      f"[{FAMILY_SPEARMAN['low']:.2f}, {FAMILY_SPEARMAN['high']:.2f}]")
display(FAMILY_TABLE[FAMILY_TABLE.family.ne('')].set_index('family')[
    ['n_mouse_hcop', 'n_human_hcop', 'detected', 'F', 'F_probe_clean', 'atlas_F', 'F_as_expected', 'clears_P_band']].round(2))

family_rows = lit[lit.route.eq('family')].copy()
family_rows['F'] = FAMILY_TABLE.F.reindex(family_rows.key.astype(int)).to_numpy()
family_rows['in_family_set'] = family_rows.key.astype(int).isin(FAMILY_SET).to_numpy()
family_rows['correct'] = np.sign(family_rows.F) == family_rows.expected_sign
family_rows.to_csv(tables / 'literature_families.csv', index=False)
display(family_rows)

## 9 · Pathways (descriptive)

- For each of the 1,513 sets of notebooks 66 and 67, the mean L over members that are reference genes.
- The atlas mean of A over the members it analyses.
- The Spearman correlation between the two, across pathways with ≥ 5 members on each side.
- The literature pathway rows (L16 and L17, lists of transporters) are checked by the sign of their members'
  mean L. Their rank among the 1,513 pathway means is given, where 1 means most human-higher. This check is
  descriptive.
- No pathway is tested.

In [ ]:
pathway_rows = []
for pathway_id, members in GENE_SETS.items():
    idx = position_of.reindex(list(members)).dropna().astype(int).to_numpy()
    ours = idx[REF[idx]]
    atlas_ = idx[ANALYSED[idx] & np.isfinite(A[idx])]
    pathway_rows.append({'pathway_id': pathway_id, 'members': len(idx), 'members_L': len(ours),
                         'mean_L': L[ours].mean() if len(ours) else np.nan, 'members_A': len(atlas_),
                         'mean_A': A[atlas_].mean() if len(atlas_) else np.nan})
PATHWAYS = pd.DataFrame(pathway_rows).set_index('pathway_id')
PATHWAYS['rank_L'] = PATHWAYS.mean_L.rank(ascending=False)
PATHWAYS.to_csv(tables / 'pathways.csv')
both_sides = PATHWAYS.members_L.ge(5) & PATHWAYS.members_A.ge(5)
PATHWAY_SPEARMAN = bootstrap_spearman(PATHWAYS.mean_L[both_sides], PATHWAYS.mean_A[both_sides], n_boot=N_BOOT, seed=SEED)
print(f"Pathway means, L vs A over {PATHWAY_SPEARMAN['n']} pathways: Spearman {PATHWAY_SPEARMAN['rho']:.2f} "
      f"[{PATHWAY_SPEARMAN['low']:.2f}, {PATHWAY_SPEARMAN['high']:.2f}] (gene sets overlap; descriptive)")
literature_pathways = lit[lit.route.eq('pathway')].copy()
pathway_values = []
for row in literature_pathways.itertuples():
    mouse_list = [g.strip() for g in str(row.mouse_genes).split(';')]
    human_list = [g.strip() for g in str(row.human_genes).split(';')]
    pairs_ = zip(mouse_list, human_list) if len(mouse_list) == len(human_list) else ((g, '') for g in mouse_list)
    ours_ = sorted({g for g in (our_gene(m, h) for m, h in pairs_) if g is not None})
    idx = position_of.reindex(ours_).to_numpy().astype(int)
    on_ref, on_atlas = idx[REF[idx]], idx[ANALYSED[idx] & np.isfinite(A[idx])]
    mean_l = L[on_ref].mean() if len(on_ref) else np.nan
    pathway_values.append({'members': len(mouse_list), 'members_in_universe': len(idx), 'members_L': len(on_ref),
                           'mean_L': mean_l, 'mean_A': A[on_atlas].mean() if len(on_atlas) else np.nan,
                           'rank_among_pathways': int(1 + (PATHWAYS.mean_L > mean_l).sum()) if len(on_ref) else np.nan,
                           'members_L_with_expected_sign': int((np.sign(L[on_ref]) == row.expected_sign).sum())})
literature_pathways = pd.concat([literature_pathways.reset_index(drop=True), pd.DataFrame(pathway_values)], axis=1)
literature_pathways['correct'] = np.sign(literature_pathways.mean_L) == literature_pathways.expected_sign
literature_pathways.to_csv(tables / 'literature_pathways.csv', index=False)
display(literature_pathways)

## 10 · Criteria and reading

The criteria and readings were fixed in the protocol. A criterion that is not evaluable counts as failed.
**A pass licenses no gene-level claim.** Level differences are reported with the housekeeping band and the pair
spread, never with a p-value.

In [ ]:
READINGS = {   # the protocol's readings table, verbatim
    ('pass', 'pass'): 'Level is a usable descriptive measurement.',
    ('fail', 'pass'): 'Level agrees with the atlas genome-wide but not with the curated controls, which are protein '
                      'measurements. List the failing controls, their atlas sign and their probe balance.',
    ('pass', 'fail'): "The agreement rests on a few curated genes and does not hold genome-wide. Level stays out of the "
                      "paper's claims.",
    ('fail', 'fail'): "Level differences in our data are not interpretable. The paper keeps the SI's statement."}
CRITERIA = pd.DataFrame({'(i) literature controls, correct sign': CRITERION_I,
                         '(ii) atlas-strong genes with |L| >= 1, sign agreement': CRITERION_II}).T
CRITERIA['threshold'] = PASS_SHARE
CRITERIA['minimum n'] = [MIN_LITERATURE, MIN_ATLAS_GENES]
outcome = tuple('pass' if s == 'pass' else 'fail' for s in CRITERIA.status)
VERDICT = ('level recovers known biology' if outcome == ('pass', 'pass')
           else 'the overall criterion is not met: level does not recover known biology')
CRITERIA.to_csv(tables / 'criteria.csv')
display(CRITERIA)
print(f'Verdict: {VERDICT}. Reading ({outcome[0]}, {outcome[1]}): {READINGS[outcome]}')
# The reading asks for the failing controls with their atlas sign and probe balance.
FAILING = evaluable[~evaluable.correct][['control_id', 'key', 'expected', 'confidence', 'value', 'pair_min', 'pair_max',
                                        'A', 'A_q', 'atlas_sign_as_expected', 'mouse_probes', 'human_probes',
                                        'equal_probes', 'nearly_absent']].sort_values('value')
FAILING.to_csv(tables / 'failing_literature_controls.csv', index=False)
display(FAILING.round(2))
print(f"(iii) Spearman L vs A {SPEARMAN.loc['L vs A (criterion iii)', 'rho']:.2f}; atlas ceiling "
      f"{SPEARMAN.loc['atlas ceiling: A (Lake) vs A (Census human)', 'rho']:.2f}. "
      f"(iv) families in the expected direction: {FAMILY_SUMMARY.loc['F in the expected direction', 'share']:.0%} of "
      f"{int(FAMILY_SUMMARY.loc['F in the expected direction', 'n'])}.")

## 11 · Main figure

- **(a)** The primary literature controls (verified for mouse). Each shows L (P for separated genes) with its
  range over the four section × mouse pairs, against the housekeeping band. Colour and marker give the correct
  or opposite sign. Labels give the expected species: (h) human higher, (m) mouse higher.
- **(b)** L against A for every analysed reference gene, with the atlas-strong genes highlighted. Those with
  |L| ≥ 1 decide (ii) and are marked by agreement.
- **(c)** Criterion (ii) sign agreement with Wilson intervals and the 80% line (top), and the (iii) Spearman
  with bootstrap intervals (bottom), under every sensitivity.
- **(d)** Family sums: F against the atlas F, split by the species with more HCOP members. The pre-named
  families are labelled.

In [ ]:
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(180 * MM, 200 * MM))
layout = GridSpec(2, 2, figure=fig, height_ratios=[1.15, 1], wspace=.3, hspace=.3)
LIMIT = 15                                # (b): values beyond +-LIMIT are drawn at the edge

# (a) literature controls
ax = fig.add_subplot(layout[0, 0])
shown = evaluable.sort_values('value').reset_index(drop=True)
ax.axvspan(*HK_BAND['L'], color=LIGHT, alpha=.8, lw=0, label='housekeeping band (L)')
ax.axvline(0, color=MUTED, lw=.5)
for i, r in shown.iterrows():
    ax.plot([r.pair_min, r.pair_max], [i, i], color=MUTED, lw=.8, solid_capstyle='round')
    ok = bool(r.correct)
    ax.scatter(r.value, i, s=16, color=AGREE_COLOR if ok else DISAGREE_COLOR, marker='o' if ok else 'X', linewidths=0,
               zorder=3)
ax.set_yticks(range(len(shown)))
ax.set_yticklabels([f"{r.key} ({'h' if r.expected_sign > 0 else 'm'})" for r in shown.itertuples()])
ax.set_ylim(-.8, len(shown) - .2)
ax.set_xlabel('Our level L (log2, centred)')
ax.scatter([], [], s=16, color=AGREE_COLOR, marker='o', label='expected sign')
ax.scatter([], [], s=16, color=DISAGREE_COLOR, marker='X', label='opposite sign')
ax.plot([], [], color=MUTED, lw=.8, label='range over the 4 pairs')
ax.legend(loc='upper left')
ax.set_title(f"a  Literature controls: {CRITERION_I['agree']}/{CRITERION_I['n']} correct", loc='left')

# (b) L against A
ax = fig.add_subplot(layout[0, 1])
base = REF & ANALYSED & np.isfinite(A)
decide = STRONG & CONVERGED & (MAGNITUDE >= LEVEL_MIN) & np.isfinite(DIRECTION)
agree = decide & (np.sign(DIRECTION) == np.sign(A))
x_shown, y_shown = np.clip(A, -LIMIT, LIMIT), np.clip(DIRECTION, -LIMIT, LIMIT)
ax.scatter(x_shown[base & ~STRONG], y_shown[base & ~STRONG], s=1.5, color=GREY, alpha=.3, linewidths=0, rasterized=True,
           label='analysed genes')
ax.scatter(x_shown[base & STRONG & ~decide], y_shown[base & STRONG & ~decide], s=2.5, color=MUTED, alpha=.6,
           linewidths=0, rasterized=True, label='atlas-strong, |L| < 1')
ax.scatter(x_shown[agree], y_shown[agree], s=4, color=AGREE_COLOR, marker='o', linewidths=0, rasterized=True,
           label='atlas-strong, |L| ≥ 1: same sign')
ax.scatter(x_shown[decide & ~agree], y_shown[decide & ~agree], s=7, color=DISAGREE_COLOR, marker='X', linewidths=0,
           rasterized=True, label='atlas-strong, |L| ≥ 1: opposite sign')
ax.set_xlim(-LIMIT - .5, LIMIT + .5)
ax.set_ylim(-LIMIT - .5, LIMIT + .5)
for v in (-LEVEL_MIN, LEVEL_MIN):
    ax.axhline(v, color=LIGHT, lw=.6, zorder=0)
    ax.axvline(v, color=LIGHT, lw=.6, zorder=0)
ax.axhline(0, color=MUTED, lw=.5, zorder=0)
ax.axvline(0, color=MUTED, lw=.5, zorder=0)
ax.set_xlabel('Atlas level A (log2, centred)')
ax.set_ylabel('Our level L (log2, centred)')
outside = int(((np.abs(A) > LIMIT) | (np.abs(DIRECTION) > LIMIT))[base | decide].sum())
ax.set_title(f"b  (ii) {CRITERION_II['agree']:,}/{CRITERION_II['n']:,} same sign", loc='left')
ax.text(.98, .02, f'{outside} genes beyond ±{LIMIT} drawn at the edge', transform=ax.transAxes, ha='right',
        va='bottom', fontsize=5, color=MUTED)
ax.legend(loc='upper left', markerscale=2.5, fontsize=5)

# (c) criterion (ii) and (iii) under every sensitivity, labels on the left half of the cell
panel_c = layout[1, 0].subgridspec(2, 2, width_ratios=[1.25, 1], height_ratios=[1.1, 1], hspace=.45)
ax = fig.add_subplot(panel_c[0, 1])
rows_c = AGREEMENT.iloc[::-1]
y = np.arange(len(rows_c))
ax.errorbar(rows_c.share.astype(float), y, xerr=[rows_c.share.astype(float) - rows_c.wilson_low.astype(float),
                                                 rows_c.wilson_high.astype(float) - rows_c.share.astype(float)],
            fmt='o', ms=3, color=INK, ecolor=MUTED, elinewidth=.8, capsize=0)
ax.axvline(PASS_SHARE, color=DISAGREE_COLOR, lw=.8, ls='--')
ax.set_yticks(y)
ax.set_yticklabels([f'{name} (n = {int(n):,})' for name, n in zip(rows_c.index, rows_c.n)], fontsize=5)
ax.set_xlabel('(ii) share with the atlas sign')
ax.text(-1.35, 1.06, 'c  Sensitivities', transform=ax.transAxes, fontweight='bold', fontsize=6.5, va='bottom')
ax = fig.add_subplot(panel_c[1, 1])
rows_s = SPEARMAN.drop(index=[i for i in SPEARMAN.index if i.startswith('probe diagnostic')]).iloc[::-1]
y = np.arange(len(rows_s))
ax.errorbar(rows_s.rho.astype(float), y, xerr=[rows_s.rho.astype(float) - rows_s.low.astype(float),
                                               rows_s.high.astype(float) - rows_s.rho.astype(float)],
            fmt='o', ms=3, color=INK, ecolor=MUTED, elinewidth=.8, capsize=0)
ax.axvline(0, color=MUTED, lw=.5)
ax.set_yticks(y)
ax.set_yticklabels([name.replace('atlas ceiling: ', 'ceiling: ') for name in rows_s.index], fontsize=5)
ax.set_xlabel('(iii) Spearman ρ (95% bootstrap interval)')

# (d) families
ax = fig.add_subplot(layout[1, 1])
shown_f = FAMILY_TABLE[use_atlas]
for sign_, color, marker, label in ((-1, DISAGREE_COLOR, 's', 'more members in mouse'),
                                    (1, AGREE_COLOR, 'o', 'more members in human')):
    part = shown_f[shown_f.expected_sign.eq(sign_)]
    ax.scatter(part.atlas_F, part.F, s=7, color=color, marker=marker, linewidths=0, alpha=.8, label=label)
for _, r in shown_f[shown_f.family.ne('')].iterrows():
    ax.annotate(r.family, (r.atlas_F, r.F), xytext=(3, 2), textcoords='offset points', fontsize=5, color=INK)
ax.axhline(0, color=MUTED, lw=.5)
ax.axvline(0, color=MUTED, lw=.5)
ax.set_xlabel('Atlas family level (log2, centred)')
ax.set_ylabel('Our family level F (log2, centred)')
expected_row = FAMILY_SUMMARY.loc['F in the expected direction']
ax.set_title(f"d  Families: {int(expected_row.agree)}/{int(expected_row.n)} as expected; "
             f"ρ with atlas {FAMILY_SPEARMAN['rho']:.2f}", loc='left')
ax.legend(loc='lower right', fontsize=5)
save(fig, 'fig1_level_positive_control')
plt.show()

## 12 · Post hoc (added after the criteria were seen)

This section is not part of the protocol. Nothing here changes a criterion or the reading.

- **(a)** Do the failing controls fail in the atlas too? This compares our sign with the atlas sign for the
  primary controls. Where the atlas (mRNA, no probes) also disagrees with a protein-based control, the
  disagreement is not specific to our measurement.
- **(b)** The pre-named families whose F and atlas F have opposite signs. Each member is listed with its probe
  count and our detection (notebook 62's inventories).

In [ ]:
post_controls = evaluable.assign(
    L_sign_as_atlas=np.sign(evaluable.value) == np.sign(evaluable.A),
    group=np.where(evaluable.correct, 'correct sign',
                   np.where(evaluable.atlas_sign_as_expected, 'opposite sign; the atlas agrees with the control',
                            'opposite sign; the atlas also disagrees with the control')))
POST_HOC_CONTROLS = post_controls.groupby('group').agg(
    controls=('key', 'size'), same_sign_as_atlas=('L_sign_as_atlas', 'sum'), genes=('key', lambda g: ', '.join(sorted(g))))
POST_HOC_CONTROLS.loc['all primary controls'] = [len(post_controls), int(post_controls.L_sign_as_atlas.sum()), '']
POST_HOC_CONTROLS.to_csv(tables / 'posthoc_controls_vs_atlas.csv')
display(POST_HOC_CONTROLS)

probe_detail = {sp: pd.read_csv(nb62 / 'tables' / f'gene_inventory_{sp}.csv',
                                usecols=['gene', 'component', 'ortholog_class', 'n_probes', 'probe_status', 'detection_1',
                                         'detection_2']) for sp in ('mouse', 'human')}
family_probe_rows = []
for component, r in FAMILY_TABLE[FAMILY_TABLE.family.ne('')].iterrows():
    for sp in ('mouse', 'human'):
        t = probe_detail[sp]
        t = t[t.component.eq(component) & t.ortholog_class.isin(FAMILY_CLASSES)]
        family_probe_rows.append({
            'family': r.family, 'species': sp, 'F': r.F, 'atlas_F': r.atlas_F,
            'opposite_sign': bool(np.sign(r.F) != np.sign(r.atlas_F)),
            'members (probes, mean detection)': ', '.join(f'{g} ({p}, {(d1 + d2) / 2:.1%})' for g, p, d1, d2
                                                          in zip(t.gene, t.n_probes, t.detection_1, t.detection_2))})
POST_HOC_FAMILIES = pd.DataFrame(family_probe_rows)
POST_HOC_FAMILIES.to_csv(tables / 'posthoc_prenamed_family_probes.csv', index=False)
display(POST_HOC_FAMILIES[POST_HOC_FAMILIES.opposite_sign].round(2))

## 13 · Run record

In [ ]:
import platform

from rpy2.robjects import r as R

cache_entries = []
for sidecar in sorted(cache.glob('*.key.json'), key=lambda p: json.loads(p.read_text())['written_utc']):
    meta = json.loads(sidecar.read_text())
    if meta['stage'] in SECONDS:
        cache_entries.append({'stage': meta['stage'], 'key': meta['key'], 'written_utc': meta['written_utc'],
                              'compute seconds': round(SECONDS[meta['stage']], 1)})
CACHE_TABLE = pd.DataFrame(cache_entries).drop_duplicates('stage', keep='last').sort_values('stage').reset_index(drop=True)
display(CACHE_TABLE)
record = {
    'notebook_logic_version': NOTEBOOK_LOGIC_VERSION,
    'protocol': 'docs/results/pt-gene-model-nb.md, Notebook 68 protocol (commit aa15069)',
    'frozen_inputs': {'literature': {'file': str(LITERATURE.relative_to(project)), 'sha256': LITERATURE_SHA256,
                                     'expected_sign': EXPECTED_SIGN},
                      'housekeeping': {'file': 'external/housekeeping/HK_genes.txt', 'sha256': HOUSEKEEPING_SHA256}},
    'inputs': INPUT_TABLE.to_dict(), 'n_jobs': N_JOBS,
    'gene_model': {'module': 'pseudospace.pathway_pipelines.standard_gene_model', 'basis_df': BASIS_DF,
                   'grid_points': GRID_POINTS, 'grid': [float(MODEL.grid[0]), float(MODEL.grid[-1])],
                   'grid_weights': 'uniform (1/101)', 'nb_gam': nb_gam.GENE_MODEL},
    'statistics': {'L': '(1/ln 2) uniform grid mean of Delta(x) - median over reference genes at x',
                   'P': 'equal-segment pseudobulk log2((y + 0.5) / (N + 1) 1e6), human - mouse, median-centred',
                   'P_centre': P_CENTRE, 'reference_genes': int(REF.sum()),
                   'separated': int(SEPARATED.sum()), 'not_converged': int((~CONVERGED).sum())},
    'atlas': {'donors': DONOR_TABLE.to_dict(orient='records'), 'fits': ATLAS_TABLE.to_dict(orient='records'),
              'limma': 'lmFit(~ species) + eBayes(trend = TRUE, robust = TRUE); centred at the median; BH',
              'strong': {'fdr': ATLAS_FDR, 'abs_log2': ATLAS_LFC}, 'min_nuclei': MIN_NUCLEI,
              'adult_stages': ADULT_STAGES, 'census_human_weights': CENSUS_HUMAN_WEIGHTS},
    'bands': BAND_TABLE.reset_index().to_dict(orient='records'),
    'criterion_i': LIT_TABLE.reset_index().to_dict(orient='records'),
    'criterion_ii': AGREEMENT.reset_index().to_dict(orient='records'),
    'criterion_iii': SPEARMAN.reset_index().to_dict(orient='records'),
    'families': {'summary': FAMILY_SUMMARY.reset_index().to_dict(orient='records'), 'spearman_with_atlas': FAMILY_SPEARMAN,
                 'orthogroups': len(FAMILY_SET)},
    'pathways': {'spearman': PATHWAY_SPEARMAN, 'min_members': 5},
    'criteria': CRITERIA.reset_index().to_dict(orient='records'), 'verdict': VERDICT, 'reading': READINGS[outcome],
    'failing_controls': FAILING.to_dict(orient='records'),
    'post_hoc': {'controls_vs_atlas': POST_HOC_CONTROLS.reset_index().to_dict(orient='records'),
                 'prenamed_family_probes': POST_HOC_FAMILIES.to_dict(orient='records')},
    'guards': guard_log,
    'versions': {'python': platform.python_version(), 'R': str(R('R.version.string')[0]),
                 'limma': str(R('as.character(packageVersion("limma"))')[0]),
                 **{m: version(m) for m in ('numpy', 'scipy', 'pandas', 'anndata', 'h5py', 'rpy2', 'matplotlib')}},
    'stage_cache': CACHE_TABLE.to_dict(orient='records'), 'compute_seconds_total': float(sum(SECONDS.values())),
    'wall_seconds': time.perf_counter() - NOTEBOOK_START}
(output / 'run_record.json').write_text(json.dumps(record, indent=2, default=str))
print(f"Wall time {record['wall_seconds']:.0f} s; summed uncached compute time of the cached stages "
      f"{record['compute_seconds_total']:.0f} s")